# Lab 3: The Perceptron

This lab implements the perceptron, a linear binary classifier that updates its parameters from individual misclassified examples.

The perceptron is implemented, trained on a linearly separable dataset, trained on data whose class overlap is gradually increased to observe the failure case, applied to a real, higher-dimensional dataset, and finally combined with a fixed random layer in the style of the original Mark I Perceptron.

## Learning objectives

By the end of this lab, you should be able to:
- Read and write a small class: `__init__`, attributes stored on `self`, and methods
- State the perceptron's decision rule and its update rule
- Implement both in a small class
- Write a training loop and plot the number of mistakes per epoch
- Explain, using a concrete example, why a perceptron cannot solve every classification problem
- Apply the same implementation, unmodified, to real, higher-dimensional data
- Implement a fixed random feature layer and explain what it changes about what the perceptron can separate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append("..")

np.random.seed(0)

---
## 1. Toy dataset

A single cloud of 2D points is generated below from one Gaussian distribution centered at the origin. Each point is labeled by which side of the diagonal line `x1 + x2 = 0` it falls on: `+1` if `x1 + x2 >= 0`, `-1` otherwise. Each point is then nudged slightly away from that line, in the direction its own label points, so the two classes do not touch exactly at the boundary.

The boundary runs straight through the densest part of the data, so many points end up close to it. That gives the true boundary a long reach across the whole cloud, and leaves little room for a separating line to be off.

In [ ]:
n_samples = 200

X = np.random.randn(n_samples, 2) * 3.0
y = np.where(X[:, 0] + X[:, 1] >= 0, 1.0, -1.0)
X[y == 1] += [.2, .2]
X[y == -1] -= [.2, .2]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("class counts: +1 ->", int((y == 1).sum()), " -1 ->", int((y == -1).sum()))

**Exercise:** write a function `plot_data(X, y)` that draws a scatter plot of the two features, with the points of each class in a different color, a legend, and axis labels. Use matplotlib (`plt`).

In [ ]:
def plot_data(X, y):
    # your code here
    ...


plot_data(X, y)
plt.show()

The data is linearly separable by construction: the exact line `x1 + x2 = 0` perfectly separates the two labels. Plenty of points sit close to that line, though, so a separating line that is only roughly right will still misclassify some of them.

---
## 2. The decision rule

A perceptron holds a weight vector `w` and a bias `b`. Given an input `x`, it computes a single number, the **score**:

```
score = w . x + b
```

where `.` is the dot product. The perceptron's prediction is the sign of that score: `+1` if the score is zero or positive, `-1` otherwise.

Geometrically, `w . x + b = 0` is a straight line (in 2D; a plane or hyperplane in higher dimensions). Points on one side score positive, points on the other side score negative. Training a perceptron means finding a `w` and `b` that place this line correctly.

The perceptron is written as a **module**: an object that holds parameters and provides methods that use them. Deep learning libraries also call these objects layers. A small example, unrelated to the perceptron:

```python
class Counter:
    def __init__(self, start):
        self.count = start

    def increment(self):
        self.count = self.count + 1
        return self.count


counter = Counter(start=10)
counter.increment()   # returns 11
counter.increment()   # returns 12
```

- `Counter(start=10)` creates a new object and automatically runs its `__init__` method. `__init__` sets up the object's parameters, here by storing `start` as `self.count`.
- `self` is the object itself. Anything stored on `self` (an attribute) stays available for as long as the object exists, which is how a later method call sees what an earlier one changed.
- Methods such as `increment` are functions defined inside the class. Their first argument is always `self`, and they read and modify attributes through it.
- Each object has its own attributes: a second `Counter` would have its own separate `count`.

`Perceptron` in `minitorch/layer.py` follows the same pattern. Its `__init__` is already written and creates `self.weight` (all zeros, one entry per feature) and `self.bias` (zero). The methods to implement are `predict` and `learn`, starting with `predict`, which should follow the rule above. Edits to files in `minitorch/` are only picked up after restarting the kernel and re-running the cells from the top.

A freshly created perceptron has zero weight and bias, so every score is exactly `0` and every prediction is `+1` (a score of exactly zero counts as positive). The cell below checks that, along with a positive score, a negative score, and the exact boundary case, each set up directly so the check does not depend on `learn()`. Passing it is evidence that `predict()` handles these cases, not a guarantee that it is correct in general.

**Exercise:** write the `predict` method in `minitorch/layers.py` and test it with the cell below. Don't forget to restart the kernel to load the updated code.

In [ ]:
from minitorch.layer import Perceptron

model = Perceptron(num_features=2)
assert model.predict(np.array([1.0, 1.0])) == 1, "a freshly created perceptron (weight=0, bias=0) should predict +1 for every input"

# set weight and bias directly, bypassing learn(), to check predict() in isolation
model.weight = np.array([1.0, 0.0])
model.bias = 0.0
assert model.predict(np.array([5.0, 100.0])) == 1, "expected +1 for a clearly positive score"
assert model.predict(np.array([-5.0, 100.0])) == -1, "expected -1 for a clearly negative score"
assert model.predict(np.array([0.0, 100.0])) == 1, "a score of exactly 0 should count as +1"

print("predict() looks correct on these cases.")

### Visualizing the decision boundary

The helper below draws the line `w . x + b = 0` as a solid line, and shades the region where the score is positive (the side the model predicts `+1`).

In [ ]:
def plot_decision_boundary(model, X, y, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    ax.scatter(X[y == 1, 0], X[y == 1, 1], color="tab:blue", label="+1")
    ax.scatter(X[y == -1, 0], X[y == -1, 1], color="tab:red", label="-1")
    ax.legend()
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 200),
        np.linspace(x2_min, x2_max, 200),
    )
    scores = model.weight[0] * xx1 + model.weight[1] * xx2 + model.bias

    upper = max(scores.max(), 0) + 1.0
    ax.contourf(xx1, xx2, scores, levels=[0, upper], colors=["tab:blue"], alpha=0.15)
    ax.contour(xx1, xx2, scores, levels=[0], colors="black", linewidths=2)

    return ax

In [ ]:
untrained_model = Perceptron(num_features=2)

plot_decision_boundary(untrained_model, X, y)
plt.show()

**Question:** What does the plot above mean? Why is this the case?

---
## 3. The update rule

`predict` does not update the parameters, it only reports the current `w` and `b`. `learn` updates the parameters when a prediction is incorrect.

The rule, when `predict(x)` does **not** match the true label `y`:
```
w = w + y * x
b = b + y
```
and no change occurs if the prediction was already correct.

If the true label is `+1` and the prediction was `-1` (the score was too low), adding `x` to `w` increases the score for that `x` on a subsequent evaluation. If the true label is `-1` and the prediction was `+1` (the score was too high), subtracting `x` decreases the score.

Implement `learn` in `minitorch/layer.py`. After saving the file, restart the kernel and re-run the cells from the top so the notebook picks up the change. The cell below checks `learn` on a few cases.

In [ ]:
model = Perceptron(num_features=2)

x = np.array([1.0, 2.0])
label = -1  # a fresh perceptron predicts +1, so this is guaranteed to be a mistake

updated = model.learn(x, label)

assert updated is True, "learn() should report that it updated the weights"
assert np.array_equal(model.weight, np.array([-1.0, -2.0])), f"expected weight = [-1, -2], got {model.weight}"
assert model.bias == -1.0, f"expected bias = -1.0, got {model.bias}"

# the same point should now be classified correctly, so learning from it again
# should NOT change the parameters a second time
updated_again = model.learn(x, label)
assert updated_again is False, "learn() should not update the weights when the prediction is already correct"
assert np.array_equal(model.weight, np.array([-1.0, -2.0])), "weights changed on a call that should not have updated anything"
assert model.bias == -1.0, "bias changed on a call that should not have updated anything"

print("learn() looks correct on these cases.")

### Watching updates

The cell below picks one random example from the dataset, shows the decision boundary before and after `learn` is called on that example, and highlights the example that was used. Run the setup cell once to get a fresh model, then run the following cell repeatedly, for example 20 to 30 times.

In [ ]:
def training_step(model, X, y):
    idx = np.random.randint(len(X))
    xi, yi = X[idx], y[idx]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    plot_decision_boundary(model, X, y, ax=axes[0])
    axes[0].scatter(xi[0], xi[1], s=250, facecolors="none", edgecolors="black", linewidths=2)
    axes[0].set_title("before")

    updated = model.learn(xi, yi)

    plot_decision_boundary(model, X, y, ax=axes[1])
    axes[1].scatter(xi[0], xi[1], s=250, facecolors="none", edgecolors="black", linewidths=2)
    axes[1].set_title("after (updated)" if updated else "after (no update needed)")

    plt.show()
    plt.close(fig)  # this cell is meant to be re-run many times in place; close each figure so they do not pile up
    return updated

In [ ]:
# run this cell once to (re)start from a fresh model
step_model = Perceptron(num_features=2)

In [ ]:
# re-run this cell repeatedly
training_step(step_model, X, y)

The circled point is the example `learn` was just called on. The title of the right panel says whether the parameters changed.

---
## 4. Training on the full dataset

Watching one update at a time is slow. Training automates it: `learn` is called on every example in the dataset, and this is repeated for several **epochs** (full passes through the data). Counting the mistakes made in each epoch shows how training progresses.

**Exercise:** write a function `train(model, X, y, epochs)` that does this and returns a list containing the number of mistakes made in each epoch. `model.learn` returns `True` when it updated the parameters, which means the example was a mistake.

In [ ]:
def train(model, X, y, epochs):
    # your code here
    ...

The cell below checks `train` on a two-point dataset small enough to follow by hand.

In [ ]:
x_check = np.array([[1.0], [-1.0]])
y_check = np.array([1.0, -1.0])
model_check = Perceptron(num_features=1)

result = train(model_check, x_check, y_check, epochs=3)

assert result is not None, "train() should return a list with the number of mistakes in each epoch"
assert list(result) == [1, 0, 0], f"expected [1, 0, 0] mistakes per epoch, got {list(result)}"
assert np.array_equal(model_check.weight, np.array([1.0])) and model_check.bias == -1.0, "train() should leave the model trained"

print("train() looks correct on this case.")

**Exercise:** train a new `Perceptron` on `X` and `y` for 10 epochs, storing it in `model`, and plot the number of mistakes in each epoch, with axis labels and a title.

In [ ]:
# your code here

The decision boundary of the trained model is plotted below.

In [ ]:
plot_decision_boundary(model, X, y)
plt.show()

When the number of mistakes per epoch reaches `0`, the perceptron has converged: its boundary separates the data.

**Question:** once an epoch has zero mistakes, every later epoch also has zero mistakes. What guarantees that?

---
## 5. Increasing class overlap

The dataset used so far, although its boundary runs through dense, close-together points, is still linearly separable by construction: the rule that generated the labels is itself a straight line. This section uses a different dataset, generated instead as two separate clusters, with an adjustable spread, `cluster_std`, controlling how tightly each cluster is packed around its center.

Run the cell below with the current `cluster_std = 0.6`, then keep increasing it and re-running. What happens to the scatter plot as `cluster_std` grows? Why?

In [ ]:
cluster_std = 0.6  # keep increasing this and re-running
n_per_class = 50

np.random.seed(0)
X_overlap = np.concatenate([
    np.random.randn(n_per_class, 2) * cluster_std + np.array([2.0, 2.0]),
    np.random.randn(n_per_class, 2) * cluster_std + np.array([-2.0, -2.0]),
])
y_overlap = np.concatenate([np.ones(n_per_class), -np.ones(n_per_class)])

plot_data(X_overlap, y_overlap)
plt.title(f"cluster_std = {cluster_std}")
plt.show()

Train a perceptron on this dataset and plot mistakes per epoch, exactly as in section 4.

In [ ]:
model_overlap = Perceptron(num_features=2)
mistakes_overlap = train(model_overlap, X_overlap, y_overlap, epochs=100)

plt.plot(mistakes_overlap)
plt.xlabel("epoch")
plt.ylabel("mistakes")
plt.title(f"Perceptron training, cluster_std = {cluster_std}")
plt.show()

plot_decision_boundary(model_overlap, X_overlap, y_overlap)
plt.show()

**Exercise:** go back to the `cluster_std = 0.6` cell above, increase the value, and re-run that cell and the two cells after it (or use "Run All Below"). Keep increasing `cluster_std` and repeating.

**Questions:** For each value, do mistakes per epoch reach `0` and stay there, or do they keep occurring every epoch? At some point the behavior changes sharply, so at roughly what value? What property of the data decides whether a perceptron can converge, and why does increasing `cluster_std` break it? What happens after the regime changes?

---
## 6. Real data: handwritten digits

Every dataset so far has been synthetic, 2D, and generated to make a specific point. This section uses a real dataset instead: 8x8 grayscale images of handwritten digits (0 through 9), 1797 of them, from scikit-learn's bundled `digits` dataset. Each image is a 64-dimensional input (8 x 8 pixels, flattened), a large jump up from the 2 features used everywhere else in this lab.

This section uses scikit-learn only to load the dataset, nothing else. If it is not yet installed, run this in a terminal (with the `dl-labs` environment activated), then restart the kernel:
```powershell
conda install -c conda-forge scikit-learn
```

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("data shape:", digits.data.shape)
print("images shape:", digits.images.shape)
print("labels:", sorted(set(digits.target)))

`digits.images` holds each sample as an 8x8 array (for viewing); `digits.data` holds the exact same values already flattened to a 64-element vector, using the `reshape` skill from Lab 2. A handful of examples are shown below.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap="gray_r")
    ax.set_title(str(digits.target[i]))
    ax.axis("off")
plt.show()

### A binary task: is this digit 5 or more?

Digits `5, 6, 7, 8, 9` are labeled `+1`; digits `0, 1, 2, 3, 4` are labeled `-1`. This splits the dataset into two almost exactly equal halves.

In [ ]:
X_digits = digits.data
y_digits = np.where(digits.target >= 5, 1.0, -1.0)

print("n samples:", len(X_digits))
print("num features:", X_digits.shape[1])
print("class counts: >=5 ->", int((y_digits == 1).sum()), " <5 ->", int((y_digits == -1).sum()))

The same `Perceptron` class and the same `train` function from section 4 are reused here unchanged, just applied to 64-dimensional real data instead of 2-dimensional synthetic data.

In [ ]:
model_digits = Perceptron(num_features=X_digits.shape[1])
mistakes_digits = train(model_digits, X_digits, y_digits, epochs=150)

plt.plot(mistakes_digits)
plt.xlabel("epoch")
plt.ylabel("mistakes")
plt.title("Perceptron training on digits: is this digit 5 or more?")
plt.show()

final_mistakes = sum(1 for xi, yi in zip(X_digits, y_digits) if model_digits.predict(xi) != yi)
final_accuracy = 1 - final_mistakes / len(X_digits)
print(f"final misclassified: {final_mistakes} / {len(X_digits)}")
print(f"final accuracy: {final_accuracy:.4f}")

Unlike the separable datasets earlier in this lab, mistakes per epoch never reach `0`. The count drops for the first several epochs, then settles into oscillating around some fixed level for the rest of training, the same signature of non-convergence seen in section 5 once the clusters started to overlap. The final accuracy is still fairly high (typically around 85 to 86 percent), since many digits are easy to place correctly, but the perceptron never finds a single straight line (technically, a hyperplane in 64 dimensions) that gets all of them right, because no such line exists for this labeling.

**Question:** Do you think training a perceptron to separate only 0 and 1 would be easier?
Can you think of a way to set the weights by hand to solve the task?
How would you check you hypothesis?

In [ ]:
# Optional: validate you hypothesis here

---
## 7. Fixed random connections

Rosenblatt's original Mark I Perceptron did not feed raw pixel values into its trainable layer at all. Between the camera and the trainable weights sat a layer of "association units": each one wired to a random subset of pixels, with fixed (never trained) random weights, followed by a threshold. Only the final layer, on top of that fixed random layer, was ever trained.

This section rebuilds that structure as a second module. `RandomProjection` in `minitorch/layer.py` has two methods to implement:

- `__init__(num_inputs, num_outputs)` creates the layer's weight, a matrix with `num_inputs` rows and `num_outputs` columns of random values following a standard normal distribution. It is generated once, when the layer is created, and never changed afterward.
- `transform(x)` projects a batch of inputs through the weight and thresholds the result at `0`, producing binary features.

Implement `__init__` first. The cell below checks it.

In [ ]:
from minitorch.layer import RandomProjection

layer = RandomProjection(num_inputs=4, num_outputs=10)

assert layer.weight.shape == (4, 10), f"expected weight shape (4, 10), got {layer.weight.shape}"
assert layer.weight.min() < 0 < layer.weight.max(), "the weight should contain both negative and positive values"
assert not np.array_equal(layer.weight, RandomProjection(num_inputs=4, num_outputs=10).weight), "two layers should not share the same random weight"

np.random.seed(0)
first = RandomProjection(num_inputs=4, num_outputs=10).weight
np.random.seed(0)
second = RandomProjection(num_inputs=4, num_outputs=10).weight
assert np.array_equal(first, second), "the weight should come from NumPy's random generator, so that np.random.seed makes it reproducible"

large = RandomProjection(num_inputs=500, num_outputs=500).weight
assert abs(large.mean()) < 0.05 and abs(large.std() - 1) < 0.1, "the weight values should follow a standard normal distribution (mean 0, standard deviation 1)"

print("__init__ looks correct on these cases.")

Then implement `transform`. The cell below checks it.

In [ ]:
layer = RandomProjection(num_inputs=2, num_outputs=3)
layer.weight = np.array([[1.0, -1.0, 0.0], [0.0, 1.0, -1.0]])  # set directly, bypassing randomness, to check in isolation

x_check = np.array([[1.0, 1.0], [-1.0, -1.0], [0.0, 0.0]])
out = layer.transform(x_check)

assert out.shape == (3, 3), f"expected shape (3, 3), got {out.shape}"
assert set(np.unique(out)).issubset({0.0, 1.0}), "transform() should only ever output 0.0 or 1.0"
expected = np.array([
    [1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 1.0],
])
assert np.array_equal(out, expected), f"expected {expected.tolist()}, got {out.tolist()}"

print("transform() looks correct on this case.")

### Exploring the number of random features

Once both methods work, the same `Perceptron` class can train on the layer's output instead of raw pixels, exactly as in section 6, with no changes to `Perceptron` itself. The cell below builds a `RandomProjection` with `num_random_features` outputs, transforms the digit data through it, and trains a perceptron on the result, plotting mistakes per epoch exactly as in section 4.

Run it with the current `num_random_features = 64`, then keep increasing the value and re-running. What happens to the mistakes-per-epoch curve? Why might that be?

In [ ]:
num_random_features = 64  # keep increasing this and re-running

np.random.seed(0)
layer = RandomProjection(num_inputs=X_digits.shape[1], num_outputs=num_random_features)
A_digits = layer.transform(X_digits)

model_random = Perceptron(num_features=num_random_features)
mistakes_random = train(model_random, A_digits, y_digits, epochs=150)

plt.plot(mistakes_random)
plt.xlabel("epoch")
plt.ylabel("mistakes")
plt.title(f"Perceptron on {num_random_features} random features")
plt.show()

print("mistakes in the final epoch:", mistakes_random[-1])

**Question:** Do you think increasing the number of random features is a good idea?

---
## Running the tests

Once every exercise above is complete, run the full test suite from a terminal, from the repository's root folder, with the `dl-labs` environment activated:
```powershell
python -m pytest tests/test_layer.py -v
```

The suite covers `Perceptron.predict`, `Perceptron.learn`, `RandomProjection.__init__` and `RandomProjection.transform` more thoroughly than the self-checks in the notebook. If a test fails, its name indicates which method to revisit.

---
## Wrap-up

This lab implemented a perceptron as a module: `__init__` sets up its weight and bias, `predict` applies a linear decision rule, and `learn` updates the parameters from misclassified examples. A training loop counted mistakes per epoch. The perceptron converges when a straight line can separate the two classes, visible as mistakes per epoch dropping to zero and staying there. As the labeling rule stops corresponding to any straight line at all, as with grouping digits by numeric value, that line stops existing, and mistakes per epoch never reach zero no matter how long training continues. The same class, unmodified, was applied directly to real, high-dimensional data, and then to that same data passed through a second module, a fixed random feature layer, which turned a labeling rule the perceptron could not separate on raw pixels into one it could separate perfectly, given enough random features.